In [2]:
from src.data import DataPipeline
from src.train import conv_rnn_training
from src.dataset import SARDataset
import torch
from torch.utils.data import random_split
import logging
from pathlib import Path
import os

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger(__name__)

In [4]:
# Data dirs
DATA_DIR = "data/preprocessed"
LABEL_PATH = "data/labels/sample_2018_2024_normalized.tif"
WEIGHTS_DIR = Path("checkpoints")
LOGS_DIR = Path("logs")

# Data loader sizes
PATCH_SIZE = 128
BATCH_SIZE = 2

# Hyperparameters
HIDDEN_CHANNELS = [16, 32, 64]
HEAD_CHANNELS = ([16], [32, 16])
LEARNING_RATE = [1e-3, 1e-2]

# Epochs
EPOCHS = 50

# Constants
KERNEL_SIZE = 3
VAL_FRACTION = 0.2
POS_WEIGHT = 3.26
INPUT_CHANNELS = 2
SEED = 0

In [5]:
def fetch_data(skip_raw=True, skip_preproc=True):
    # Data engineering
    pipeline = DataPipeline()
    if not skip_raw:
        pipeline.raw()
    if not skip_preproc:
        pipeline.preprocessed()
        pipeline.labels()
        
fetch_data()

In [ ]:
def train():
    
    # Training
    logger.info("Training procedure started.")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        logger.info(f"Device: {torch.cuda.get_device_name(0)}")
        
    dataset = SARDataset(data_dir=DATA_DIR, label_path=LABEL_PATH, patch_size=PATCH_SIZE)

    n_val = max(1, int(len(dataset) * VAL_FRACTION))
    n_train = len(dataset) - n_val
    generator = torch.Generator().manual_seed(SEED)
    train_set, val_set = random_split(dataset, [n_train, n_val], generator=generator)
       
    WEIGHTS_DIR.mkdir(exist_ok=True)
    LOGS_DIR.mkdir(exist_ok=True)

    for hidden in HIDDEN_CHANNELS:
        for head in HEAD_CHANNELS:
            for lr in LEARNING_RATE:
                weights_str = f"hidden{hidden}-" + f"head{len(head)}-" + f"lr{lr}".replace(".", "p")
                weights_path = WEIGHTS_DIR / f"{weights_str}.pth"
                history_path = LOGS_DIR / f"{weights_str}.csv"
                
                if weights_path.exists():
                    continue
                
                history = conv_rnn_training(
                    input_channels=INPUT_CHANNELS,
                    kernel_size=KERNEL_SIZE,
                    hidden_channels=hidden,
                    head_channels=head,
                    lrate=lr,
                    train_dataset=train_set,
                    val_dataset=val_set,
                    batch_size=BATCH_SIZE,
                    epochs=EPOCHS,
                    pos_weight=POS_WEIGHT,
                    device=device,
                    weights_path=weights_path,
                    arch="ConvGRU",
                    patience=8
                )
                
                history.to_parquet(history_path, index=False)